In [ ]:
import pyautogui
from pynput.mouse import Controller, Button

In [ ]:
screenwidth, screenheight = pyautogui.size()

In [4]:
mouse = Controller()

In [5]:
mouse.press(Button.left)
mouse.release(Button.left)

In [6]:
mouse.press(Button.right)
mouse.release(Button.right)

In [37]:
import mediapipe as mp
import cv2
import pyautogui
from pynput.mouse import Controller, Button
import time

In [38]:
mp_hand=mp.solutions.hands
hand=mp_hand.Hands(max_num_hands=1)
mp_drawing=mp.solutions.drawing_utils

In [39]:
def count_fingers(landmarks):
    tip=[8, 12, 16, 20]
    pip=[6, 10, 14, 18]
    c=0
    for t,p in zip(tip,pip):
        t_x,t_y=landmarks.landmark[t].x, landmarks.landmark[t].y
        p_x,p_y=landmarks.landmark[p].x, landmarks.landmark[p].y
        if t_y<p_y:
            c+=1
    pinky_tx=landmarks.landmark[20].x
    thumb_tx=landmarks.landmark[4].x
    thumb_px=landmarks.landmark[2].x
    # right hand
    if thumb_tx > pinky_tx:
        if thumb_tx>thumb_px:
            c+=1
    else:
        if thumb_tx<thumb_px:
            c+=1
    return c

In [40]:
def index_finger_tip(landmarks, screenwidth, screenheight):
    tx, ty = round(landmarks.landmark[8].x*screenwidth), round(landmarks.landmark[8].y*screenheight)
    return (tx, ty)

In [41]:
mouse = Controller()
def detect_gestures(landmarks):
    screenwidth, screenheight = pyautogui.size()
    ty, py = landmarks.landmark[8].y, landmarks.landmark[6].y
    mty, mpy = landmarks.landmark[12].y, landmarks.landmark[10].y
    rty, rpy = landmarks.landmark[16].y, landmarks.landmark[14].y
    if count_fingers(landmarks)==1 and ty<py:
        x, y = index_finger_tip(landmarks, screenwidth, screenheight)
        pyautogui.moveTo(x, y)
    elif count_fingers(landmarks)==2 and ty<py and mpy<mty:
        mouse.click(Button.left, 1)
        time.sleep(0.1)
    elif count_fingers(landmarks)==3 and ty<py and mpy<mty and rty<rpy:
        pyautogui.doubleClick()

In [36]:
cap=cv2.VideoCapture(0)
while True:
    ret, frame=cap.read()
    if not ret:
        print('no frame')
        break
    # height, width, _ = frame.shape
    # frame=cv2.flip(frame, 1)
    rgb_frame=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    # rgb_frame=cv2.resize(rgb_frame, (800, 600))
    res=hand.process(rgb_frame).multi_hand_landmarks
    if res:
        landmarks=res[0]
        cv2.putText(frame, f'{len(res)} hands detected', (100, 30), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 23, 123), 1)
        detect_gestures(landmarks)
        # c=0
        # for i in res:
        #     mp_drawing.draw_landmarks(frame, i, mp_hand.HAND_CONNECTIONS)
        #     c+=count_fingers(i)
        # cv2.putText(frame, f'{c} fingers open', (100, 60), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 23, 123), 1)
        # if peace(i):
        #     cv2.putText(frame, 'Peace!', (100, 120), cv2.FONT_HERSHEY_COMPLEX, 1, (0, 100, 0), 1)
    else:
        cv2.putText(frame, 'No hands detected', (100, 30), cv2.FONT_HERSHEY_COMPLEX, 1, (255, 23, 123), 1)
    # frame=cv2.Canny(frame, 200, 200, cv2.THRESH_BINARY)
    cv2.imshow('VirtualFrame', frame)
    if cv2.waitKey(3)==ord('a'):
        cv2.destroyAllWindows()
        cap.release()
        break